# 层归一化

本教程将编写高性能的层归一化内核，目标是比 PyTorch 实现运行得更快。

你将学习：

- 使用 Triton 实现反向传播。
- 使用 Triton 实现并行归约。
---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 动机

LayerNorm 最初由参考文献 [BA2016] 提出，用于改善序列模型（例如 Transformer）或小批量神经网络的表现。

它接受向量 $x$ 并输出同形状的向量 $y$。首先从输入中减去均值，再除以标准差；随后应用由可学习权重 $w$ 和偏置 $b$ 构成的仿射变换。前向计算为：

$$
y=\frac{x-\mathrm{E}[x]}{\sqrt{\mathrm{Var}(x)+\epsilon}}\odot w+b
$$

其中 $\epsilon$ 是为数值稳定性而加入分母的小常数。

首先来看前向传播实现。

## 反向传播

层归一化的反向传播比前向计算更复杂。令 $\hat{x}$ 为仿射变换前的归一化输入：

$$
\hat{x}=\frac{x-\mathrm{E}[x]}{\sqrt{\mathrm{Var}(x)+\epsilon}}
$$

输入 $x$ 的向量—雅可比积（VJP）为：

$$
\nabla_x=\frac{1}{\sigma}
\left(
\nabla_y\odot w
-\underbrace{\left(\frac{1}{N}\hat{x}\cdot(\nabla_y\odot w)\right)}_{c_1}\hat{x}
-\underbrace{\frac{1}{N}\nabla_y\cdot w}_{c_2}
\right)
$$

这里 $\odot$ 表示逐元素乘法，$\cdot$ 表示点积；$\sigma$ 对应前向归一化中的分母，即 $\sqrt{\mathrm{Var}(x)+\epsilon}$。中间常数 $c_1$ 和 $c_2$ 用于提高实现的可读性。

权重和偏置对单行输入的梯度较为直接：

$$
\nabla_w=\nabla_y\odot\hat{x},
\qquad
\nabla_b=\nabla_y
$$

由于同一批次的所有行共享权重 $w$ 和偏置 $b$，还需要对各行的贡献求和。

为高效完成这一操作，教程采用并行归约：每个内核实例把若干行的部分梯度累加到 GROUP_SIZE_M 个独立缓冲区之一。这些缓冲区保留在 L2 缓存中，再由另一个函数进一步归约，得到最终梯度。

原文用输入行数 $M=4$、缓冲区数为 2 的示例解释这一过程，示意图在本笔记中省略。

第一阶段，共享同一缓冲区的行需要通过锁，保证同一时刻只有一个内核实例写入。第二阶段，对缓冲区继续归约，得到最终的 $\nabla_w$ 和 $\nabla_b$。

对应实现中，第一阶段由函数 _layer_norm_bwd_dx_fused 完成，第二阶段由函数 _layer_norm_bwd_dwdb 完成。

## 性能基准测试

现在可以比较自定义内核与 PyTorch 的性能。这里关注每行特征数据小于 64 KB 的输入。

将测试模式设为反向传播，即可评估反向计算的性能。

## 参考文献

- [BA2016] Jimmy Lei Ba、Jamie Ryan Kiros、Geoffrey E. Hinton，*Layer Normalization*（层归一化），arXiv，2016。